*  Will clean this up a bit but just want to get something posted to
the git for the day.
*   Still fairly inconsitient will fix with independent K-trials and a more intelligent cloud model.

But currently im happy with the results on the benchmark question "Will the UK National Threat Level reduce from SEVERE before September 2026?" courtesy metaculus. It comes to some great points of reasioning with a reasonable probability estimate.

In [1]:
!sudo apt-get update && sudo apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama ddgs pydantic

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,797 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,049 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,258 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://securi

with langchain

In [2]:
!pip install langgraph langchain-ollama duckduckgo-search pydantic

In [18]:
import os
import sys
import shutil
import warnings
import subprocess
import time
import json
import urllib.request
import urllib.error
import argparse
from datetime import datetime
from typing import Literal, TypedDict, List, Optional

for var in ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY']:
    val = os.environ.get(var)
    if val and not val.startswith('http'):
        os.environ[var] = f"http://{val}"

os.environ['NO_PROXY'] = '127.0.0.1,localhost,' + os.environ.get('NO_PROXY', '')
os.environ['no_proxy'] = '127.0.0.1,localhost,' + os.environ.get('no_proxy', '')

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

from langgraph.graph import StateGraph, START, END
from ollama import Client
from ddgs import DDGS
from pydantic import BaseModel, Field

CURRENT_DATE = datetime.now().strftime("%B %d, %Y")
#shoutout tetlock
TEMPORAL_PREAMBLE = f"""Today is {CURRENT_DATE}.
When evaluating evidence, treat all dates in the context relative to TODAY.

CRITICAL SOURCE SKEPTICISM & SANITY CHECKS:
1. The web is filled with AI-generated SEO spam and fake speculative news.
2. You must rigorously evaluate the credibility of the sources provided. Be highly skeptical of crypto blogs, unknown tech sites, and speculative opinion pieces.
3. SANITY CHECK: You MUST evaluate if the claims mathematically and economically make sense. For example, an $800B+ valuation for an early-stage AI startup is an immediate red flag for fake news/hallucination, regardless of the URL.
4. Only treat information from highly reputable mainstream or financial news outlets (e.g., Bloomberg, Reuters, WSJ, SEC.gov) as verified facts, and ONLY if the numbers pass the sanity check.

FORECASTING BEST PRACTICES (FERMI & BAYESIAN):
1. Base Rates First: Always consider the 'Reference Class' (similar historical precedents) to establish a baseline probability (The Prior).
2. Bayesian Updating: Treat new, specific evidence as updates that adjust the probability up or down from the Base Rate.
3. Fermi Estimation: Break down complex unknowns into smaller, estimable parts."""

class AppConfig:
    """Holds the runtime configuration for the CLI or interactive session."""
    def __init__(self, question: str, provider: str, model: str, iterations: int):
        self.question = question
        self.provider = provider.lower()
        self.model = model
        self.max_iterations = iterations

# Global config instance to be initialized in main
config = None


def setup_ollama(model_name: str):
    print(f"[*] Checking hardware acceleration for local Ollama")
    ollama_path = shutil.which("ollama") or "/usr/local/bin/ollama"
    server_env = os.environ.copy()
    server_env["OLLAMA_HOST"] = "127.0.0.1:11434"

    try:
        gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        if gpu_check.returncode == 0:
            print(" ↳ [✓] NVIDIA GPU detected.")
            if "/usr/lib64-nvidia" not in server_env.get("LD_LIBRARY_PATH", ""):
                server_env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + server_env.get("LD_LIBRARY_PATH", "")
        else:
            print(" ↳ [!] No GPU found. Running on CPU mode.")
    except Exception:
        print(" ↳ [!] Could not query GPU details. Proceeding on standard mode.")

    print("[*] Starting Ollama server...")
    subprocess.Popen([ollama_path, "serve"], env=server_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    server_ready = False
    for _ in range(15):
        try:
            req = urllib.request.Request("http://127.0.0.1:11434/")
            urllib.request.urlopen(req, timeout=2)
            server_ready = True
            print(" ↳ [✓] Server is up and responding.")
            break
        except Exception:
            time.sleep(1)

    if server_ready:
        print(f"[*] Pulling '{model_name}' (this takes a moment if not cached)...")
        subprocess.run([ollama_path, "pull", model_name], env=server_env, check=True)
        print(" ↳ [✓] Local Model ready.\n")
    else:
        print(" ↳ [X] ERROR: Server did not start properly.")

class SearchQuery(BaseModel):
    scratchpad: str = Field(description="Think out loud about the best keywords to use before writing the query.")
    query: str = Field(description="A highly targeted 1 to 2 sentence search query. No quotes or punctuation.")

class AgentBeliefState(BaseModel):
    scratchpad: str = Field(description="Use this space to brainstorm and outline your analysis before formalizing it.")
    sanity_check: str = Field(description="Critically evaluate if the claims, numbers, and valuations in the text actually make economic/physical sense, or if they are likely fake news.")
    step_by_step_thinking: str = Field(description="Perform a highly detailed, multi-step critical analysis of the text. Break down the nuances, assess the weight of each source.")
    evidence_points: list[str] = Field(description="A comprehensive, exhaustive list of all specific data points, quotes, and factual evidence found in the text.")
    confidence: Literal["Weak", "Moderate", "Strong"]
    reasoning: str = Field(description="Comprehensive summary of why this confidence level was chosen based on the extracted evidence. Defend your stance thoroughly.")

class GapAnalysis(BaseModel):
    scratchpad: str = Field(description="Evaluate the current evidence and decide if there are critical gaps before filling out the rest.")
    requires_more_info: bool = Field(description="True if critical context is missing and a search is required. False if ready to forecast.")
    information_gaps: list[str] = Field(description="List of specific missing facts, recent events, or data points needed.")
    search_queries: list[str] = Field(description="1 to 2 targeted search queries to resolve the gaps. Empty if no info needed.")
    reasoning: str = Field(description="Explanation of why more information is or isn't needed.")

class SynthesisState(BaseModel):
    scratchpad: str = Field(description="Draft your thoughts on reference classes, base rates, and how the evidence shifts the probability.")
    sanity_check: str = Field(description="Perform a final sanity check on the evidence. Throw out any claims that violate economic or physical reality.")
    reference_class: str = Field(description="Identify the most similar historical category or precedent for this event.")
    base_rate_probability: int = Field(description="The historical percentage rate (0-100) of this reference class succeeding without new specific evidence (The Prior).")
    bayesian_update_analysis: str = Field(description="Detailed analysis of how the specific Thesis and Antithesis evidence shifts the base rate up or down.")
    dialectic_resolution: str = Field(description="How you reconcile the clashes, contradictions, and data gaps between both sides.")
    aufhebung_future_state: str = Field(description="Sublation: A linguistic representation of the world/situation after the event.")
    final_probability: int = Field(description="The final calibrated probability percentage as a WHOLE NUMBER from 0 to 100 (e.g., 45, NOT 0.45).")

class GraphState(TypedDict):
    question: str
    iteration: int
    thesis_context: str
    antithesis_context: str
    search_queries: List[str]
    thesis_state: Optional[AgentBeliefState]
    antithesis_state: Optional[AgentBeliefState]
    synthesis_state: Optional[SynthesisState]

ollama_client = Client(host="http://127.0.0.1:11434")

def get_json_template(schema_class):
    schema_dict = schema_class.model_json_schema()
    template = {}
    for prop_name, prop_details in schema_dict.get("properties", {}).items():
        desc = prop_details.get("description", "")
        prop_type = prop_details.get("type", "string")
        if prop_type == "array":
            template[prop_name] = [f"<{desc}>"]
        elif prop_type in ["integer", "number"]:
            template[prop_name] = 85
        elif "enum" in prop_details:
            template[prop_name] = f"<Must be one of: {prop_details['enum']}>"
        else:
            template[prop_name] = f"<{desc}>"
    return json.dumps(template, indent=2)

def extract_json(raw_text):
    text = raw_text.strip()
    end_fence = text.rfind('```')
    if end_fence != -1:
        start_fence = text.rfind('```', 0, end_fence)
        if start_fence != -1:
            block = text[start_fence + 3 : end_fence].strip()
            if block.lower().startswith('json'):
                block = block[4:].strip()
            return block
    start_idx = text.find('{')
    end_idx = text.rfind('}')
    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        return text[start_idx:end_idx+1]
    return text

def generate_with_ollama(prompt, schema_class, max_retries, options):
    template_str = get_json_template(schema_class)
    full_prompt = f"{prompt}\n\nOUTPUT INSTRUCTIONS:\nYou must format your response as a valid JSON object matching this EXACT template. Do NOT wrap it in 'properties' or 'required' keys. Just fill in the blanks:\n```json\n{template_str}\n```\nCrucially, you must place your final JSON object inside a ```json ... ``` code block at the very end of your response."

    for attempt in range(1, max_retries + 1):
        try:
            print(f" ↳ [Streaming Local] {schema_class.__name__}: ", end="", flush=True)
            response_stream = ollama_client.chat(model=config.model, messages=[{'role': 'user', 'content': full_prompt}], stream=True, options=options)
            content_acc = ""
            for chunk in response_stream:
                msg = chunk.get('message', {}) if isinstance(chunk, dict) else getattr(chunk, 'message', {})
                c_chunk = msg.get('content', '') if isinstance(msg, dict) else getattr(msg, 'content', '')
                if c_chunk:
                    print(c_chunk, end="", flush=True)
                    content_acc += c_chunk
            print()

            clean_json_str = extract_json(content_acc.strip())
            return schema_class.model_validate_json(clean_json_str)
        except Exception as e:
            print(f"\n ↳ [!] Retry {attempt}/{max_retries} failed ({e}). Retrying in 2s...")
            time.sleep(2)
    return None

def generate_with_cloud_api(prompt, schema_class, max_retries):
    """Dynamically routes to the correct LangChain integration for Cloud APIs."""
    from langchain_core.messages import HumanMessage

    if config.provider == "openai":
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model=config.model, temperature=0.3)
    elif config.provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        llm = ChatAnthropic(model_name=config.model, temperature=0.3)
    elif config.provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        llm = ChatGoogleGenerativeAI(model=config.model, temperature=0.3)
    else:
        raise ValueError(f"Unsupported cloud provider: {config.provider}")

    # Cloud APIs support native structured outputs reliably
    structured_llm = llm.with_structured_output(schema_class)
    messages = [HumanMessage(content=prompt)]

    for attempt in range(1, max_retries + 1):
        try:
            print(f" ↳ [Cloud API] Requesting {schema_class.__name__} from {config.provider.upper()}... ", end="", flush=True)
            response = structured_llm.invoke(messages)
            print("[✓]")
            return response
        except Exception as e:
            print(f"\n ↳ [!] API Retry {attempt}/{max_retries} failed ({e}). Retrying in 2s...")
            time.sleep(2)
    return None

def generate_with_retries(prompt, schema_class, max_retries=3, options=None):
    """Main routing wrapper for local vs API generation."""
    if options is None:
        options = {"temperature": 0.3, "num_predict": 8192, "num_ctx": 65536}

    if config.provider == "ollama":
        return generate_with_ollama(prompt, schema_class, max_retries, options)
    else:
        return generate_with_cloud_api(prompt, schema_class, max_retries)

def get_search_context(query: str, fallback_query: str, max_chars: int = 12000) -> str:
    """Pulls search snippets via DDGS."""
    if not query or not query.strip():
        query = fallback_query

    print(f" ↳ Searching: '{query}'")
    try:
        results = DDGS().text(query, max_results=15)
    except Exception as e:
        print(f" ↳ [!] Web search failed ({e}). Retrying with fallback...")
        try:
            results = DDGS().text(fallback_query, max_results=15)
        except Exception:
            return "No recent web search results could be retrieved."

    formatted_snippets = []
    source_count = 1
    for r in results:
        if source_count > 6:
            break
        body = r.get('body', '').strip()
        link = r.get('href', 'Unknown URL')

        if "youtube.com" in link or "youtu.be" in link: continue
        if body:
            print(f"    ✓ Scraped: {link}")
            formatted_snippets.append(f"[Source: {link}]\n[Snippet {source_count}]: {body}")
            source_count += 1

    combined_context = "\n\n".join(formatted_snippets)
    if len(combined_context) > max_chars:
        combined_context = combined_context[:max_chars] + "... [Context Truncated]"

    return combined_context if combined_context.strip() else "No actionable context found."

def node_init_search(state: GraphState) -> GraphState:
    print("\n[*] Pre-Flight: Initial Adversarial Intelligence Gathering")
    question = state["question"]

    t_prompt = f"{TEMPORAL_PREAMBLE}\nYou are an expert web-searching assistant. Target Question: '{question}'\nGenerate a highly targeted 1 to 2 sentence search query to find evidence that this WILL happen."
    t_res = generate_with_retries(t_prompt, SearchQuery, options={"temperature": 0.3, "num_predict": 2048, "num_ctx": 32768})
    t_query_str = t_res.query if t_res else question + " success"

    a_prompt = f"{TEMPORAL_PREAMBLE}\nYou are an expert web-searching assistant. Target Question: '{question}'\nGenerate a highly targeted 1 to 2 sentence search query to find evidence that this WILL NOT happen or will fail."
    a_res = generate_with_retries(a_prompt, SearchQuery, options={"temperature": 0.3, "num_predict": 2048, "num_ctx": 32768})
    a_query_str = a_res.query if a_res else question + " failure"

    print("\n[*] Fetching Initial Context...")
    thesis_ctx = get_search_context(t_query_str, question + " success")
    antithesis_ctx = get_search_context(a_query_str, question + " failure")

    return {"thesis_context": thesis_ctx, "antithesis_context": antithesis_ctx, "iteration": 1}

def node_thesis_agent(state: GraphState) -> GraphState:
    print(f"\n[↻] --- Iteration {state['iteration']} / {config.max_iterations} ---")
    print("[+] Generating Thesis Response...")
    prompt = f"{TEMPORAL_PREAMBLE}\nYou are the Thesis Agent. Your job is to make the strongest possible case for why the event WILL happen.\nTarget Question: {state['question']}\nContext:\n{state['thesis_context']}"

    thesis_state = generate_with_retries(prompt, AgentBeliefState)
    if not thesis_state:
        thesis_state = AgentBeliefState(scratchpad="Error", sanity_check="Error", step_by_step_thinking="Failed to parse.", evidence_points=["Error"], confidence="Moderate", reasoning="Fallback adopted.")
    return {"thesis_state": thesis_state}

def node_antithesis_agent(state: GraphState) -> GraphState:
    print("[+] Generating Antithesis Response...")
    prompt = f"{TEMPORAL_PREAMBLE}\nYou are the Antithesis Agent. Your job is to make the strongest possible case for why the event WILL NOT happen (or will be delayed).\nTarget Question: {state['question']}\nContext:\n{state['antithesis_context']}"

    antithesis_state = generate_with_retries(prompt, AgentBeliefState)
    if not antithesis_state:
        antithesis_state = AgentBeliefState(scratchpad="Error", sanity_check="Error", step_by_step_thinking="Failed to parse.", evidence_points=["Error"], confidence="Moderate", reasoning="Fallback adopted.")
    return {"antithesis_state": antithesis_state}

def node_judge_gap(state: GraphState) -> GraphState:
    print("[?] Running Judge Gap Analysis...")
    prompt = f"""{TEMPORAL_PREAMBLE}\nYou are the Dialectic Judge. Evaluate the current arguments regarding: '{state['question']}'\nDo you have enough verifiable facts? Or are there critical gaps requiring web searching?
THESIS EVIDENCE: {json.dumps(state['thesis_state'].evidence_points if state['thesis_state'] else [])}
ANTITHESIS EVIDENCE: {json.dumps(state['antithesis_state'].evidence_points if state['antithesis_state'] else [])}"""

    gap_analysis = generate_with_retries(prompt, GapAnalysis)
    queries = gap_analysis.search_queries if gap_analysis and gap_analysis.requires_more_info else []
    return {"search_queries": queries, "iteration": state["iteration"] + 1}

def node_web_search(state: GraphState) -> GraphState:
    print("\n[*] Executing Judge's targeted queries to fill gaps...")
    new_evidence_context = ""
    for sq in state.get("search_queries", []):
        if sq.strip():
            new_evidence_context += get_search_context(sq, sq) + "\n\n"

    update_str = f"\n\n[UPDATE - ITERATION {state['iteration'] - 1} NEW EVIDENCE FOUND]:\n{new_evidence_context}"
    return {"thesis_context": state["thesis_context"] + update_str, "antithesis_context": state["antithesis_context"] + update_str, "search_queries": []}

def node_synthesis(state: GraphState) -> GraphState:
    print("\n[★] Phase C: Final Dialectic Resolution")
    print(" ↳ Synthesizing final probability (this may take a moment)...")
    prompt = f"""{TEMPORAL_PREAMBLE}
You are the Dialectic Judge. Make sure to give a probability corresponding to this target "{state['question']} this is the question you will be giving a corresponding probaility to at the end of your thoughts, do come up with a question of your own to which you assign a probability you will be punished if you dont answer THIS question"
Step 1: Establish a Base Rate (Prior) using a historical Reference Class.
Step 2: Use the clashing arguments below as new evidence to perform a Bayesian Update.
Step 3: do sublation to transcend the clash of info. Outputs your final calibrated probability.
This needs work at this should be the final linguistic representation of the world with fuzzy sets and all, most valuable piece for RL

-this is the final thesis report
Evidence Gathered: {json.dumps(state['thesis_state'].evidence_points if state['thesis_state'] else [])}

-this is the final antithesis report
Evidence Gathered: {json.dumps(state['antithesis_state'].evidence_points if state['antithesis_state'] else [])}"""

    synthesis_state = generate_with_retries(prompt, SynthesisState, options={"temperature": 0.4, "num_predict": 8192, "num_ctx": 32768})

    if synthesis_state:
        print("\n")
        print("      DIALECTIC JUDGEMENT REPORT")
        print("")
        print(f"[Sanity Check]:\n{synthesis_state.sanity_check}\n")
        print(f"[Reference Class]:\n{synthesis_state.reference_class}\n")
        print(f"[Base Rate (Prior)]: {synthesis_state.base_rate_probability}%\n")
        print(f"[Bayesian Update Analysis]:\n{synthesis_state.bayesian_update_analysis}\n")
        print(f"[Dialectic Resolution]:\n{synthesis_state.dialectic_resolution}\n")
        print(f"[Aufhebung (Future State)]:\n{synthesis_state.aufhebung_future_state}\n")
        print("")
        print(f"DEFUZZIFIED CRISP PROBABILITY : {synthesis_state.final_probability}%")
        print("\n")

    return {"synthesis_state": synthesis_state}


# Logic Map for langgraph:
# 1. Start -> `node_init_search` Generates search queries and gets initial context
# 2. Parallel/Sequential Analysis(should be parrellel but local runs sequen) -> `node_thesis_agent`  and `node_antithesis_agent`
# 3. Gap Analysis -> `node_judge_gap` judge agent decides if theres enough info
# 4. Conditional Router -> `route_after_judge`:
#    a. If gaps exist and iterations < MAX: -> `node_web_search` -> Loops back to Agents
#    b. If satisfied or max iterations: -> `node_synthesis` -> Ends workflow

def route_after_judge(state: GraphState) -> Literal["node_web_search", "node_synthesis"]:
    """Conditional Edge: decides whether to loop back to search or move to synthesis."""
    if len(state.get("search_queries", [])) > 0 and state["iteration"] <= config.max_iterations:
        print(" ↳ Judge requested more information. Routing to Web Search.")
        return "node_web_search"

    print(" ↳ Judge is satisfied OR max iterations reached. Routing to Synthesis.")
    return "node_synthesis"

def build_graph():
    workflow = StateGraph(GraphState)
    workflow.add_node("node_init_search", node_init_search)
    workflow.add_node("node_thesis_agent", node_thesis_agent)
    workflow.add_node("node_antithesis_agent", node_antithesis_agent)
    workflow.add_node("node_judge_gap", node_judge_gap)
    workflow.add_node("node_web_search", node_web_search)
    workflow.add_node("node_synthesis", node_synthesis)

    workflow.add_edge(START, "node_init_search")
    workflow.add_edge("node_init_search", "node_thesis_agent")
    workflow.add_edge("node_thesis_agent", "node_antithesis_agent")
    workflow.add_edge("node_antithesis_agent", "node_judge_gap")

    workflow.add_conditional_edges(
        "node_judge_gap",
        route_after_judge,
        {"node_web_search": "node_web_search", "node_synthesis": "node_synthesis"}
    )

    workflow.add_edge("node_web_search", "node_thesis_agent")
    workflow.add_edge("node_synthesis", END)

    return workflow.compile()

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Hegel's Fuzzy Forecaster - Dialectic Agent Workflow")
    parser.add_argument("question", type=str, nargs='?', default="Will the UK National Threat Level reduce from SEVERE before September 2026?", help="The target forecasting question.")
    parser.add_argument("--provider", type=str, choices=["ollama", "openai", "anthropic", "gemini"], default="ollama", help="LLM provider (default: ollama).")
    parser.add_argument("--model", type=str, default="granite4.1:8b", help="Model name (e.g., granite4.1:8b, gpt-4o, claude-3-5-sonnet-20240620).")
    parser.add_argument("--iterations", type=int, default=3, help="Max gap-analysis loops (default: 3).")

    # Safe argument parsing to prevent crashes in Jupyter Notebook / interactive environments
    if hasattr(sys, 'ps1') or 'ipykernel' in sys.modules:
        print("Running in interactive mode. Using default arguments.")
        args, _ = parser.parse_known_args([])
    else:
        args = parser.parse_args()

    # Initialize Global Config
    config = AppConfig(question=args.question, provider=args.provider, model=args.model, iterations=args.iterations)

    print(f"\nTarget Forecasting Question: \"{config.question}\"\n")
    print(f"Provider: {config.provider.upper()} | Model: {config.model} | Max Loops: {config.max_iterations}\n")

    if config.provider == "ollama":
        setup_ollama(config.model)
    else:
        print(f"[*] Bypassing local Ollama setup. Preparing to call Cloud API ({config.provider.upper()}).")

    app = build_graph()
    initial_state = {
        "question": config.question,
        "iteration": 1,
        "thesis_context": "",
        "antithesis_context": "",
        "search_queries": [],
        "thesis_state": None,
        "antithesis_state": None,
        "synthesis_state": None
    }

    print("[*] Initializing Agentic Graph Workflow...")
    final_state = app.invoke(initial_state)
    print("\n[✓] Forecasting workflow completed successfully.\n")

Running in interactive mode. Using default arguments.

Target Forecasting Question: "Will the UK National Threat Level reduce from SEVERE before September 2026?"

Provider: OLLAMA | Model: granite4.1:8b | Max Loops: 3

[*] Checking hardware acceleration for local Ollama
 ↳ [✓] NVIDIA GPU detected.
[*] Starting Ollama server...
 ↳ [✓] Server is up and responding.
[*] Pulling 'granite4.1:8b' (this takes a moment if not cached)...
 ↳ [✓] Local Model ready.

[*] Initializing Agentic Graph Workflow...

[*] Pre-Flight: Initial Adversarial Intelligence Gathering
 ↳ [Streaming Local] SearchQuery: ```json
{
  "scratchpad": "To determine if the UK National Threat Level will reduce from SEVERE before September 2026, I need to focus on authoritative sources that provide updates or statements from UK government security agencies (e.g., MI5, Home Office) regarding threat assessments. Keywords should include 'UK National Threat Level', 'SEVERE', 'reduction', 'date range' up to 'September 2026'. Avoid